# 将文件切分并生成二维码

In [ ]:
pip install numpy==1.24.3 opencv-contrib-python==4.5.2.54 opencv-python==4.5.2.54 tqdm==4.65.0

In [2]:
import os
import qrcode
from PIL import Image
import base64
import cv2
from tqdm.notebook import tqdm

def split_bytes_into_batches(file_bytes, batch_size):
    batches = [file_bytes[i:i+batch_size] for i in range(0, len(file_bytes), batch_size)]
    return batches

def generate_qr_code(data, output_path):
    # 将二进制数据编码为Base64字符串
    base64_data = base64.b64encode(data).decode('utf-8')

    qr = qrcode.QRCode(
        version=None,
        error_correction=qrcode.constants.ERROR_CORRECT_L,
        box_size=10,
        border=4,
    )
    qr.add_data(base64_data)
    qr.make(fit=True)
    qr_img = qr.make_image(fill_color="black", back_color="white")
    qr_img.save(output_path)
    
def convert_file_to_qr_codes(file_path, output_directory, batch_size):
    with open(file_path, 'rb') as file:
        file_bytes = file.read()

    batches = split_bytes_into_batches(file_bytes, batch_size)

    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    for i, batch in tqdm(enumerate(batches), total=len(batches), desc="Generating QR codes"):
        output_path = os.path.join(output_directory, f'qr_code_{str(i+100000)}.png')
        generate_qr_code(batch, output_path)

    print("转换完成！")

    
    
# 二维码合成视频


def images_to_video(image_directory, output_path):
    image_files = sorted([f for f in os.listdir(image_directory) if f.endswith('.jpg') or f.endswith('.png')])

    if len(image_files) == 0:
        print("该目录中没有找到图片文件！")
        return

    # 获取图片总数量
    image_count = len(image_files)

    # 获取最大图片尺寸
    max_width = 0
    max_height = 0
    for image_file in image_files:
        image_path = os.path.join(image_directory, image_file)
        image = cv2.imread(image_path)
        height, width, _ = image.shape
        max_width = max(max_width, width)
        max_height = max(max_height, height)

    # 计算视频长度和帧率
    video_length = image_count
    fps = 12

    # 创建视频编码器
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')  # 可根据需要更改编码器
    video = cv2.VideoWriter(output_path, fourcc, fps, (max_width, max_height))
    counter = 0

    # 将每张图片逐帧写入视频

    counter = 0
    for image_file in tqdm(image_files, total=len(image_files), desc="Processing images"):
        image_path = os.path.join(image_directory, image_file)
        frame = cv2.imread(image_path)

        # 调整图片尺寸以使其与最大尺寸一致
        frame_height, frame_width, _ = frame.shape
        if frame_height != max_height or frame_width != max_width:
            frame = cv2.resize(frame, (max_width, max_height))

        video.write(frame)
        counter += 1

    # 释放视频编码器资源
    video.release()

    print("视频转换完成！")
    print(counter)
    

# 清空中间文件

def delete_all_files_in_directory(directory):
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path):
            os.remove(file_path)

In [3]:
# 每个二维码批次的字符数
batch_size = 500  
# 输入的文件路径
file_path = '/Users/huanghaozhou/Downloads/profile(4).txt.zip' 
# 输出视频的路径
output_path = '/Users/huanghaozhou/Downloads/scan_to_all_file/profile(4).mp4'  
# 中间过程
output_directory = os.path.dirname(file_path)+'/qr_code'
image_directory = output_directory


delete_all_files_in_directory(output_directory)
convert_file_to_qr_codes(file_path, output_directory, batch_size)
images_to_video(image_directory, output_path)
delete_all_files_in_directory(output_directory)


Generating QR codes:   0%|          | 0/59 [00:00<?, ?it/s]

转换完成！


Processing images:   0%|          | 0/59 [00:00<?, ?it/s]

视频转换完成！
59


# 将手机录制的视频切分成图像（Resize 并做灰度处理）

In [2]:
import cv2
import os

def split_video_to_frames(video_path, output_directory, output_resolution=(640, 480)):
    # 创建输出目录
    os.makedirs(output_directory, exist_ok=True)

    # 打开视频文件
    video = cv2.VideoCapture(video_path)

    frame_count = 0

    while video.isOpened():
        ret, frame = video.read()

        if not ret:
            break

        # 将帧转换为灰度图像
        gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # 调整图像分辨率
        resized_frame = cv2.resize(gray_frame, output_resolution)

        # 生成输出图像文件名
        output_file = os.path.join(output_directory, f'frame_{str(frame_count+100000)}.png')

        # 保存当前帧为图像文件
        cv2.imwrite(output_file, resized_frame)

        frame_count += 1

    # 关闭视频文件
    video.release()

    print(f"成功将视频切分为{frame_count}帧，并保存在{output_directory}目录中。")


# 扫码并拼合文件

In [5]:
import cv2
import os
from tqdm.notebook import tqdm
import base64
detector = cv2.wechat_qrcode_WeChatQRCode()

def process_images_in_directory(directory, output_file):
    files = os.listdir(directory)
    files.sort()
    scanned_data = []  # 创建一个列表来存储已扫描过的二维码数据
    pbar = tqdm(total=len(files), desc="Scan QR codes")
    counter = 0
    data=base64.b64decode('')
    for file in files:
        pbar.update(1)
        if file.endswith('.jpg') or file.endswith('.jpeg') or file.endswith('.png'):
            file_path = os.path.join(directory, file)
            image = cv2.imread(file_path)
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            decoded_objects, points = detector.detectAndDecode(gray)

            if len(decoded_objects) > 0:
                base64_data = decoded_objects[0]

            # 将Base64字符串解码回二进制数据
                data = base64.b64decode(base64_data)

            # 如果当前图片的扫码内容已经在之前的扫码内容中出现过，则丢弃
            if data in scanned_data:
                continue

            # 将扫码结果添加到列表中
            scanned_data.append(data)
            counter += 1
        else:
            continue

    # 将所有扫描到的数据合并为一个字节串
    merged_data = b''.join(scanned_data)

    # 将合并后的字节串写入输出文件
    with open(output_file, 'wb') as f:
        f.write(merged_data)

    print("处理完成！扫码结果已保存在" + output_file + "文件中。")
    print("共处理" + str(counter) + "条数据")




In [7]:
# 使用示例
video_path = '/Users/huanghaozhou/Downloads/IMG_7447.mp4'  # 输入的视频文件路径
output_directory = os.path.dirname(video_path)+'/output_qr_code'  # 输出图像帧的目录
output_resolution = (1280, 720) 
directory = output_directory  
output_file = '/Users/huanghaozhou/Downloads/profile_remake1.txt'  # 输出扫码结果的文件路径

split_video_to_frames(video_path, output_directory, output_resolution)
process_images_in_directory(directory, output_file)

成功将视频切分为2368帧，并保存在/Users/huanghaozhou/Downloads/output_qr_code目录中。


Scan QR codes:   0%|          | 0/2369 [00:00<?, ?it/s]

Error: Invalid base64-encoded string: number of data characters (333) cannot be 1 more than a multiple of 4

# 下方为代码备份

# 将手机录制的视频切分成图像（不做灰度处理）

In [1]:
import cv2
import os

def split_video_to_frames(video_path, output_directory):
    # 创建输出目录
    os.makedirs(output_directory, exist_ok=True)

    # 打开视频文件
    video = cv2.VideoCapture(video_path)

    frame_count = 0

    while video.isOpened():
        ret, frame = video.read()

        if not ret:
            break

        # 生成输出图像文件名
        output_file = os.path.join(output_directory, f'frame_{str(frame_count+100000)}.png')

        # 保存当前帧为图像文件
        cv2.imwrite(output_file, frame)

        frame_count += 1

    # 关闭视频文件
    video.release()

    print(f"成功将视频切分为{frame_count}帧，并保存在{output_directory}目录中。")

# 使用示例
video_path = '/Users/huanghaozhou/Downloads/scan_to_qrcode/CAM_6.MP4'  # 输入的视频文件路径
output_directory = '/Users/huanghaozhou/Downloads/scan_to_qrcode/zip_r_code_6'  # 输出图像帧的目录

split_video_to_frames(video_path, output_directory)


KeyboardInterrupt: 